# 02 RegGS Smoke Test

Goal:

1. verify the exported RegGS-ready scene can be loaded correctly;
2. verify the internal RegGS train/test split logic on that scene;
3. generate a small debug config;
4. run a minimal `run_infer.py` smoke test.

This notebook is for **connectivity / pipeline sanity**, not final experiment quality.

Supports multiple datasets: Re10k-1, DL3DV-2 (set `DATASET_NAME` below).

In [1]:
from pathlib import Path
import json
import subprocess
import shutil
import os
import sys
import copy

# ============== DATASET SELECTION ==============
# Options: 'Re10k-1', 'DL3DV-2'
DATASET_NAME = 'Re10k-1'
DATASET_NAME = 'DL3DV-2'
# ===============================================

CV_ROOT = Path('~/CV_Project').expanduser()
REGGS_ROOT = CV_ROOT / 'third_party' / 'RegGS'
PART2_ROOT = CV_ROOT / 'part2'
CONFIG_ROOT = PART2_ROOT / 'configs'
OUTPUT_ROOT = CV_ROOT / 'output' / 'part2'

# Scene path configuration based on dataset
SCENE_CONFIGS = {
    'Re10k-1': {
        'scene_name': 'reggs_re10k1_fullseq_256',
    },
    'DL3DV-2': {
        'scene_name': 'reggs_dl3dv2_fullseq_256',
    },
}

if DATASET_NAME not in SCENE_CONFIGS:
    raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}. Options: {list(SCENE_CONFIGS.keys())}")

SCENE_ROOT = CV_ROOT / 'dataset' / DATASET_NAME / 'part2' / SCENE_CONFIGS[DATASET_NAME]['scene_name']
SCENE_PARENT = SCENE_ROOT.parent
SCENE_NAME = SCENE_ROOT.name

CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('DATASET_NAME =', DATASET_NAME)
print('REGGS_ROOT =', REGGS_ROOT)
print('SCENE_ROOT =', SCENE_ROOT)
print('SCENE_NAME =', SCENE_NAME)

DATASET_NAME = DL3DV-2
REGGS_ROOT = /home/bzhang512/CV_Project/third_party/RegGS
SCENE_ROOT = /home/bzhang512/CV_Project/dataset/DL3DV-2/part2/reggs_dl3dv2_fullseq_256
SCENE_NAME = reggs_dl3dv2_fullseq_256


## 1. Basic exported-scene checks


In [2]:
assert SCENE_ROOT.exists(), f'Missing scene root: {SCENE_ROOT}'
assert (SCENE_ROOT / 'images').exists(), 'Missing images dir'
assert (SCENE_ROOT / 'intrinsics.json').exists(), 'Missing intrinsics.json'
assert (SCENE_ROOT / 'cameras.json').exists(), 'Missing cameras.json'

cameras = json.loads((SCENE_ROOT / 'cameras.json').read_text(encoding='utf-8'))
intrinsics = json.loads((SCENE_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
images = sorted((SCENE_ROOT / 'images').iterdir())

print('num images =', len(images))
print('num cameras =', len(cameras))
print('intrinsics =', intrinsics)
print('first image =', images[0].name if images else None)
print('first camera image_name =', cameras[0]['image_name'] if cameras else None)

assert len(images) == len(cameras), 'Image / camera count mismatch'
assert images[0].name == cameras[0]['image_name'], 'First exported image name mismatch'


num images = 306
num cameras = 306
intrinsics = {'fx': 0.7970605911654538, 'fy': 0.7972243051546298, 'cx': 0.5, 'cy': 0.5}
first image = 0000.png
first camera image_name = 0000.png


## 2. Load the scene through the actual RegGS dataset loader


In [3]:
sys.path.insert(0, str(REGGS_ROOT))

from src.entities.datasets import get_dataset

loader_cfg = {
    'input_path': str(SCENE_PARENT),
    'scene_name': SCENE_NAME,
    'H': 256,
    'W': 256,
    'fx': 1,
    'fy': 1,
    'cx': 1,
    'cy': 1,
    'depth_scale': 5000.0,
    'frame_limit': -1,
}

dataset = get_dataset('re10k')(loader_cfg)

print('len(dataset) =', len(dataset))
print('dataset.width, dataset.height =', dataset.width, dataset.height)
print('dataset.intrinsics =')
print(dataset.intrinsics)
print('first pose shape =', dataset.poses[0].shape)
print('first image loaded shape =', dataset[0][1].shape)


len(dataset) = 306
dataset.width, dataset.height = 256 256
dataset.intrinsics =
[[204.04752   0.      128.     ]
 [  0.      204.08942 128.     ]
 [  0.        0.        1.     ]]
first pose shape = (4, 4)
first image loaded shape = (256, 256, 3)


## 3. Reproduce RegGS split logic


In [4]:
def compute_split(n_frames, sample_rate, n_views):
    import numpy as np
    frame_ids = np.arange(n_frames)
    test_frame_ids = frame_ids[int(sample_rate / 2)::sample_rate]
    remain_frame_ids = np.array([i for i in frame_ids if i not in test_frame_ids])
    train_frame_ids = remain_frame_ids[np.linspace(0, remain_frame_ids.shape[0] - 1, n_views).astype(int)]
    return train_frame_ids, test_frame_ids

SMOKE_SAMPLE_RATE = 30
SMOKE_N_VIEWS = 8

train_ids, test_ids = compute_split(len(dataset), SMOKE_SAMPLE_RATE, SMOKE_N_VIEWS)
print('sample_rate =', SMOKE_SAMPLE_RATE)
print('n_views =', SMOKE_N_VIEWS)
print('train_ids =', train_ids.tolist())
print('test_ids (first 20) =', test_ids[:20].tolist())
print('num test ids =', len(test_ids))


sample_rate = 30
n_views = 8
train_ids = [0, 43, 87, 130, 174, 217, 261, 305]
test_ids (first 20) = [15, 45, 75, 105, 135, 165, 195, 225, 255, 285]
num test ids = 10


## 4. Create a minimal debug config


In [5]:
base_cfg_path = REGGS_ROOT / 'config' / 're10k.yaml'
base_cfg = json.loads(json.dumps(__import__('yaml').safe_load(base_cfg_path.read_text(encoding='utf-8'))))

smoke_cfg = copy.deepcopy(base_cfg)
smoke_cfg['dataset_name'] = 're10k'
smoke_cfg['n_views'] = int(SMOKE_N_VIEWS)
smoke_cfg['sample_rate'] = int(SMOKE_SAMPLE_RATE)
smoke_cfg['new_submap_every'] = 2
smoke_cfg['nopo_checkpoint'] = 're10k'
smoke_cfg['data']['input_path'] = str(SCENE_PARENT)
smoke_cfg['data']['scene_name'] = SCENE_NAME
smoke_cfg['data']['output_path'] = str(OUTPUT_ROOT / 'reggs_smoke' / SCENE_NAME)
smoke_cfg['cam']['H'] = 256
smoke_cfg['cam']['W'] = 256
smoke_cfg['cam']['fx'] = 1
smoke_cfg['cam']['fy'] = 1
smoke_cfg['cam']['cx'] = 1
smoke_cfg['cam']['cy'] = 1
smoke_cfg['frame_limit'] = -1

# Use dataset-specific config filename
smoke_cfg_path = CONFIG_ROOT / f'{DATASET_NAME.lower().replace("-", "_")}_smoke.yaml'
smoke_output_path = Path(smoke_cfg['data']['output_path'])
smoke_output_path.parent.mkdir(parents=True, exist_ok=True)

import yaml
smoke_cfg_path.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
print('wrote config to', smoke_cfg_path)
print('smoke output path =', smoke_output_path)
print(smoke_cfg_path.read_text(encoding='utf-8'))


wrote config to /home/bzhang512/CV_Project/part2/configs/dl3dv_2_smoke.yaml
smoke output path = /home/bzhang512/CV_Project/output/part2/reggs_smoke/reggs_dl3dv2_fullseq_256
dataset_name: re10k
checkpoint_path: null
frame_limit: -1
seed: 0
nopo_enable: true
n_views: 8
sample_rate: 30
nopo_checkpoint: re10k
new_submap_every: 2
aligner:
  iterations: 200
  cam_rot_lr: 0.0002
  cam_trans_lr: 0.002
  filter_alpha: false
  alpha_thre: 0.98
  soft_alpha: true
  mask_invalid_depth: false
  key_frame_pose: nopo
data:
  input_path: /home/bzhang512/CV_Project/dataset/DL3DV-2/part2
  output_path: /home/bzhang512/CV_Project/output/part2/reggs_smoke/reggs_dl3dv2_fullseq_256
  scene_name: reggs_dl3dv2_fullseq_256
cam:
  H: 256
  W: 256
  fx: 1
  fy: 1
  cx: 1
  cy: 1
  crop_edge: 0
  depth_scale: 5000.0



## 5. Optional cleanup before smoke run


In [6]:
RESET_OUTPUT = False
if RESET_OUTPUT and smoke_output_path.exists():
    shutil.rmtree(smoke_output_path)
    print('removed old smoke output:', smoke_output_path)
else:
    print('RESET_OUTPUT =', RESET_OUTPUT)


RESET_OUTPUT = False


## 6. Run minimal infer smoke test


In [10]:
RUN_SMOKE = True
SMOKE_TIMEOUT_SEC = 3000

cmd = ['python3', 'run_infer.py', str(smoke_cfg_path)]
print('cmd =', ' '.join(cmd))

if RUN_SMOKE:
    proc = subprocess.run(
        cmd,
        cwd=str(REGGS_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=SMOKE_TIMEOUT_SEC,
    )
    print(proc.stdout)
    print('returncode =', proc.returncode)
else:
    print('Set RUN_SMOKE=True to execute the smoke test.')


cmd = python3 run_infer.py /home/bzhang512/CV_Project/part2/configs/dl3dv_2_smoke.yaml
Using re10k.ckpt
/home/bzhang512/CV_Project/third_party/RegGS/src/utils/nopo_utils.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for a

## 7. Post-run checks


In [ ]:
print('output exists =', smoke_output_path.exists())
if smoke_output_path.exists():
    print('top-level output entries:')
    for p in sorted(smoke_output_path.iterdir())[:20]:
        print(' -', p.name)
else:
    print('No smoke output yet.')


output exists = True
top-level output entries:
 - config.yaml
 - estimated_c2w.ckpt
 - submaps
 - vis_align
 - vis_integrate


: 